<a href="https://colab.research.google.com/github/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_gpt_to_llama2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Converting a GPT Architecture to Llama 2**

- In this notebook, we will follow Sebastian's approach to converting the original GPT architecture into Llama 2 in a step by step fashion.
- The will primarily be on the main code and unnecessary bloat will be avoided.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

## **1. Setup Notebook**

- Setup the notebook using the `llms-from-scratch` pypi packages.

In [ ]:
!pip install llms_from_scratch --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 157.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
ERROR: Cannot install llms-from-scratch==0.1.0, llms_from_scratch, ml-dtypes==0.6.0 and tensorflow because these package versions have con

In [ ]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",
    "sentencepiece",
    "torch",
]
for p in pkgs:
    print(f"{p} version : {version(p)}")

huggingface_hub version : 1.33.0
sentencepiece version : 0.2.2
torch version : 2.11.0+cu130


In [ ]:
import torch
import torch.nn as nn

## **2. Begin Converting GPT Model to Llama**

### 2.1 Replace LayerNorm with RMSNorm Layer

- As shown in the diagram above, we will replace LayerNorm by RMSNorm.
- RMSNorm uses the root mean square to improve computational efficiency, compared to LayerNorm which normalized inputs using mean and variance.
- The RMSNorm operation is as follows, where $x$ is the input $\gamma$ is a trainable parameter (vector), and $\epsilon$ is a small constant to avoid zero-division errors:

$$y_i = \frac{x_i}{\text{RMS}(x)} \gamma_i, \quad \text{where} \quad \text{RMS}(x) = \sqrt{\epsilon + \frac{1}{n} \sum x_i^2}$$



In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps) # reciprocal sq. root
        return (x_normed * self.weight).to(dtype=x.dtype)
        torch.rsqrt()

In [ ]:
torch.manual_seed(123)

eg_batch = torch.randn(2, 3, 4)
rms_norm = RMSNorm(emb_dim=eg_batch.shape[-1])
rms_norm_pytorch = torch.nn.RMSNorm(eg_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(eg_batch), rms_norm_pytorch(eg_batch))

### **2.2 Replace GELU with SiLU Activation**

- Llama utilizes the SiLU (also known as the Swish function) activation function instead of GeLU.

$$
\text{silu}(x) = x \cdot \sigma(x), \quad \text{where} \quad \sigma(x) \text{ is the logistic sigmoid.}
$$


In [ ]:
class SiLU(nn.Module):
    def __init__(self):
        # Using cleaner syntax
        super().__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [ ]:
silu = SiLU()
silu_pytorch = torch.nn.SiLU()

assert torch.allclose(silu(eg_batch), silu_pytorch(eg_batch))

### **2.3 Update the FeedForward Module**

- Llama uses a Gates Linear Unit (GLU) variant of SiLU called SwiGLU. The structure of the FeedForward module will need to be changed to accomodated this.
- SwiGLU uses a gating mechanism in the FeedForward layer which has the following formula:

$$\text{SwiGLU}(x) = \text{SiLU}(\text{Linear}_1(x)) * (\text{Linear}_2(x))$$

- Linear Layers 1 and 2 go through an element-wise multiplication while the third layer, i.e. Layer 3,  is applied after the gated activation.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # dtype=cfg["dtype"] allows us to load the model directly in lower precision formats.
        # Llama also doesn't use any bias units
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        self.silu = SiLU()

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = self.silu(x_fc1) * x_fc2
        return self.fc3(x)


### **2.4 Implement RoPE**

- Compared to traditional "absolute" positional embeddings, Llama uses rotary position embeddings (RoPE), which enables it to capture both absolute and relative positional information simultaneously.
- RoPE can be implemented in two ways:
    - The _split-halves_ version and the _interleaved even/odd version_.
    - Both are mathematically the same as long as we pair dimensions consistently while using the same cos/sin ordering.
- The reference paper for RoPE is [RoFormer: Enhanced Transformer with Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864)
- This code uses the RoPE *split-halves* approach, similar to Hugging Face transformers ([modeling_llama.py](https://github.com/huggingface/transformers/blob/e42587f596181396e1c4b63660abf0c736b10dae/src/transformers/models/llama/modeling_llama.py#L173-L188))
- Further information about the The original paper and Meta's official Llama 2 repo, however, use the *interleaved (even/odd)* version ([llama/model.py](https://github.com/meta-llama/llama/blob/6c7fe276574e78057f917549435a2554000a876d/llama/model.py#L64-L74))

In [ ]:
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096):
    assert head_dim % 2 == 0, "head_dim must be divisible by 2 for the RoPE algorithm to work properly."

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0) # Shape: (context_length, head_dim // 2)

    # Expand the angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1) # Shape : (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be divisible by 2 for the RoPE algorithm to work properly."

    # Split x into the first half and the second half
    x1 = x[..., : head_dim // 2] # First half
    x2 = x[..., head_dim // 2 :] # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0) # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

- Sample code for applying RoPE to `q` and `k` tensors.

In [ ]:
# Settings
batch_size = 2
context_len = 5
num_heads = 4
head_dim = 16

# Instantiate RoPE params
cos, sin = precompute_rope_params(head_dim=head_dim, context_length=context_len)

# Dummy query and key tensors
torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, context_len, head_dim)
keys = torch.randn(batch_size, num_heads, context_len, head_dim)

# Apply rotary position embeddings
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)


### **2.5 Add RoPE to MultiHeadAttention module**

- Note that GPT applies the positional embeddings to the inputs, whereas Llama applies rotations to the query and the key vectors in the self- attention mechanism.
- We will be modifying the `MultiHeadAttention` class with the appropriate RoPE code. Additionally, the `qkv_bias` option will be removed and `bias=False` will be hardcoded.
- A dtype setting will be added to allow instantiation with a lower precision later.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dtype=None):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        ################## Key Modifications ###########################
        # set bias = False and dtype=dtype for all linear layers below
        ################################################################
        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype) # Linear layer to combine all the head outputs
        # self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

        cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)


    def forward(self, x):

        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads dimension is added to split the matrix
        # unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        ################## RoPE ######################
        keys = compute_rope(keys, self.cos, self.sin)
        queries = compute_rope(queries, self.cos, self.sin)
        ##############################################

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3) # dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill the attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # No dropout

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec



In [ ]:
### Sample Multihead Attention module's input

# Settings
batch_size = 1
context_len = 100
max_context_len = 4096
embed_dim = 128
num_heads = 4


example_batch = torch.randn((batch_size, context_len, embed_dim))

mha = MultiHeadAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=max_context_len,
    num_heads=num_heads
)

mha(example_batch)

del mha  # free up memory

### **2.6 Update the TransformerBlock module**

- Replace LayerNorm with RMSNorm, remove `dropout` and `qkv_bias` and add the `dtype` setting.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dtype=cfg["dtype"],
        )
        self.ff = FeedForward(cfg)

        ############# Modification ##############
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.norm2 = RMSNorm(cfg["emb_dim"])
        ##########################################

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x) # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut # Add the original input back

        return x

### **2.7 Update the model class**

- Time to update the model code surrounding the `TransformerBlock`.
- We will replace absolute positional embeddings for `RoPE`, `LayerNorm` with `RMSNorm`. `dropout` will be removed while `dtype` setting will be added.

In [ ]:
class Llama2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        # abs. pos. emb. and dropout removed
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # Layer norm removed and RMSNorm added
        self.final_norm = RMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits